In [ ]:
# зависимости
%pip install pandas matplotlib anytree networkx -q

## EDA, предобработка датафрейма

**1. Прочитаем датасет, посмотрим на данные и пропуски, отберем только релевантные для анализа колонки (с данными по категориям продуктов).**

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("samokat_esci.csv")
data.head()

,query,item_name,item_id,final_answer,category4_name,category3_name,category2_name,category1_name
0,энергетик без сахара,"энергетик самокат, без сахара, с соком малины,...",55e4ae6a298cf3f69d8e07a92ec9e6a0,e,энергетические напитки,энергетические напитки,напитки,безалкогольные напитки
1,салат из рубца,"салат кх волкова а. п. деревенский, 150 г",b429835fc94f84edf3e319f01ce35274,s,салаты с майонезом,салаты,готовые блюда,кулинария
2,барилла,"макароны barilla, пенне ригате, 450 г",a2f84d68a96b5f95e3b72ff3f664e1e6,e,фигурные,"спагетти, фигурные",макаронные изделия,бакалея
3,краска для волос орех,"спрей-уход для наращенных волос tashe 250 мл, ...",227b20c1eee7ba92f52dfaf6fe9c05c4,c,натуральные краски для волос,краска для волос,средства по уходу за волосами,уход
4,творог мя,"творог козочка с облачка, 5%, мягкий, из козье...",05180ad0ea61952acd22c82f5740c47b,e,мягкий творог,творог,кисломолочные традиционные продукты,молочная продукция


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79774 entries, 0 to 79773
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   query           79774 non-null  object
 1   item_name       79774 non-null  object
 2   item_id         79774 non-null  object
 3   final_answer    79774 non-null  object
 4   category4_name  79774 non-null  object
 5   category3_name  79727 non-null  object
 6   category2_name  79194 non-null  object
 7   category1_name  79054 non-null  object
dtypes: object(8)
memory usage: 4.9+ MB


In [5]:
# пропущенные значения
data.isna().sum()

query               0
item_name           0
item_id             0
final_answer        0
category4_name      0
category3_name     47
category2_name    580
category1_name    720
dtype: int64

In [6]:
data_categories = data[['category4_name', 'category3_name', 'category2_name', 'category1_name']]

# сгруппируем по первой категории и посмотрим, где больше всего объектов
data_categories.groupby(['category1_name'], dropna=False).count().sort_values('category3_name', ascending=False)

,category4_name,category3_name,category2_name
category1_name,,,
уход,5271,5271,5271
безалкогольные напитки,4745,4745,4745
молочная продукция,4565,4565,4565
парфюмерия и декоративная косметика,4056,4056,4056
бакалея,3938,3938,3938
кулинария,3524,3524,3524
кондитерские изделия,3120,3120,3120
гигиена,2953,2953,2953
хлеб и хлебобулочные изделия,2895,2895,2895


**2. Проанализируем, когда возникают пропущенные значения.**

In [7]:
data_categories_null = data_categories[data_categories.isna().any(axis=1)]
data_categories_null.head()

,category4_name,category3_name,category2_name,category1_name
65,материалы,имущество магазина,NaN,NaN
144,сэмплы,прочее,NaN,NaN
223,сканеры штрих-кода для внутренних нужд,ит оборудование,имущество магазина,NaN
260,комплектующие для продажи,NaN,NaN,NaN
288,мерч униформа,мерч,имущество магазина,NaN


Получается, что если путь более короткий, чем 4 категории, то колонки занимаются, начиная с category4_name.

Посмотрим на одноуровневые категории и проверим, что там находится.

In [8]:
data_categories_null[data_categories_null.isna().sum(axis=1) == 3]['category4_name'].value_counts()

category4_name
услуги                       23
комплектующие для продажи    11
собственные отходы            9
! на удаление !               3
упаковка                      1
Name: count, dtype: int64

## Дерево категорий

**Задача: визуализировать иерархию категорий. Какие категории самые крупные, как устроено дерево?**

**1. Преобразуем датафрейм.**

- Пропуски в категориях на более высоких уровнях заполняются данными с более назких
- В датафрейме хранятся только уникальные строки (наборы категорий) с указанием того, сколько раз они встретились в датафрейме.

In [9]:
data_categories_fillna = data_categories.copy()

# заполняем пропущенные категории снизу вверх
data_categories_fillna['category3_name'] = data_categories_fillna['category3_name'].fillna(data_categories_fillna['category4_name'])
data_categories_fillna['category2_name'] = data_categories_fillna['category2_name'].fillna(data_categories_fillna['category3_name'])
data_categories_fillna['category1_name'] = data_categories_fillna['category1_name'].fillna(data_categories_fillna['category2_name'])

# версия датасета с подсчетом уникальных строк
data_categories_counted = data_categories_fillna.copy()
data_categories_counted['count'] = data_categories_counted.groupby(list(data_categories.columns)).transform('size')
data_categories_counted = data_categories_counted.drop_duplicates()

data_categories_counted.head()

,category4_name,category3_name,category2_name,category1_name,count
0,энергетические напитки,энергетические напитки,напитки,безалкогольные напитки,589
1,салаты с майонезом,салаты,готовые блюда,кулинария,145
2,фигурные,"спагетти, фигурные",макаронные изделия,бакалея,228
3,натуральные краски для волос,краска для волос,средства по уходу за волосами,уход,96
4,мягкий творог,творог,кисломолочные традиционные продукты,молочная продукция,45


**2. Соберем связи по уровням категорий и построим дерево.**

In [10]:
from anytree import Node, RenderTree

def create_connections(df, cols=['category4_name', 'category3_name', 'category2_name', 'category1_name', 'count']):
    """Функция, собирающая все направленные связи по категориям"""

    connections = []

    # для каждой строки добавляем все последовательные пары с весами
    for index, row in df.iterrows():
        for i in range(1, len(cols)-1):
            # от большей категории - к меньшей - вес
            if not row[cols[i]] == row[cols[i-1]]:
                connections.append([row[cols[i]], row[cols[i-1]], row['count']])
                # print(index, row[cols[i-1]], row[cols[i]])

    connections = sorted(connections, key=lambda x: [x[0], x[1]])
    print(connections)

    # объединяем повторяющиеся ребра и суммируем значения
    merged = []
    for conn in connections:
        if not merged or merged[-1][:-1] != conn[:-1]:
            merged.append(conn)
        else:
            merged[-1][2] += conn[2]
            
    return merged


def create_tree(edges):
    """Функция, создающая дерево категорий"""
    nodes = {}

    for parent_name, child_name in edges:
        try:
            # создаем узел родителя
            if parent_name not in nodes:
                nodes[parent_name] = Node(parent_name)
            
            # создаем узел ребенка
            if child_name not in nodes:
                nodes[child_name] = Node(child_name)
            
            # назначаем связь
            nodes[child_name].parent = nodes[parent_name]
        except Exception as e:
            print(e)

    return nodes


def draw_tree(nodes):
    """Функция отрисовки дерева"""
    # ищем корневые узлы и визуализируем
    root = None
    for node in nodes.values():
        if node.is_root:
            root = node
            for pre, fill, node in RenderTree(root):
                print(f"{pre}{node.name}")
            print('\n')

In [11]:
edges = create_connections(data_categories_counted)
edges = [i[:-1] for i in edges]
tree = create_tree(edges)
draw_tree(tree)

[['it/office', 'мобильные и стационарные компьютеры', 12], ['it/office', 'мобильные и стационарные компьютеры', 11], ['it/office', 'мобильные и стационарные компьютеры', 13], ['it/office', 'мобильные и стационарные компьютеры', 4], ['it/office', 'мобильные и стационарные компьютеры', 3], ['it/office', 'мобильные и стационарные компьютеры', 2], ['it/office', 'мобильные и стационарные компьютеры', 7], ['it/office', 'мобильные и стационарные компьютеры', 4], ['it/office', 'мобильные и стационарные компьютеры', 3], ['it/office', 'мониторы', 6], ['it/office', 'оргтехника', 7], ['it/office', 'оргтехника', 3], ['it/office', 'оргтехника', 1], ['it/office', 'офисная периферия', 19], ['it/office', 'офисная периферия', 8], ['it/office', 'офисная периферия', 6], ['it/office', 'офисная периферия', 2], ['it/office', 'сетевое оборудование', 6], ['it/office', 'сетевое оборудование', 3], ['telecom', 'аудиотехника для гаджетов', 48], ['telecom', 'аудиотехника для гаджетов', 28], ['telecom', 'аудиотехник

**3. Визулизируем дерево в графовом формате.**